# 06 — Transformer Block — Solution

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from src.models.attention import CausalSelfAttention
from src.models.feedforward import FeedForward
from src.models.transformer import TransformerLM, TransformerConfig

torch.manual_seed(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — RMSNorm

In [ ]:
class MyRMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))   # γ

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


norm = MyRMSNorm(64)
x_t = torch.randn(2, 10, 64)
out_t = norm(x_t)
print('RMSNorm output:', out_t.shape)

# Verify: library implementation should match
from src.models.transformer_block import RMSNorm
lib_norm = RMSNorm(64)
lib_out  = lib_norm(x_t)
print('Matches library:', torch.allclose(out_t.detach(), lib_out.detach(), atol=1e-5))

## Part B — Pre-LN Decoder Block

In [ ]:
class MyDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.0):
        super().__init__()
        self.norm1 = MyRMSNorm(d_model)
        self.norm2 = MyRMSNorm(d_model)
        self.attn  = CausalSelfAttention(d_model, n_heads, dropout=dropout)
        self.ffn   = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x):
        # Pre-LN attention with residual
        attn_out, _ = self.attn(self.norm1(x))
        x = x + attn_out
        # Pre-LN FFN with residual
        x = x + self.ffn(self.norm2(x))
        return x


block = MyDecoderBlock(64, 4, 256)
x = torch.randn(2, 16, 64)
print('Block output:', block(x).shape)

## Part C — Residual Ablation

In [ ]:
class BlockNoResidual(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = CausalSelfAttention(d_model, n_heads)
        self.ffn   = FeedForward(d_model, d_ff)

    def forward(self, x):
        x, _ = self.attn(self.norm1(x))
        x = self.ffn(self.norm2(x))
        return x


def stack_and_measure_grad(block_cls, n_layers=8, d_model=64, **kwargs):
    layers = nn.Sequential(*[block_cls(d_model, **kwargs) for _ in range(n_layers)])
    embed  = nn.Embedding(100, d_model)
    x      = torch.randint(0, 100, (1, 16))
    out    = layers(embed(x))
    out.sum().backward()
    return embed.weight.grad.norm().item()


with_res = stack_and_measure_grad(MyDecoderBlock,  n_layers=8, n_heads=4, d_ff=256)
no_res   = stack_and_measure_grad(BlockNoResidual, n_layers=8, n_heads=4)

print(f'Gradient norm WITH residuals   : {with_res:.4f}')
print(f'Gradient norm WITHOUT residuals: {no_res:.6f}')
print('\nWithout residuals, gradients collapse by the time they reach the embedding.')

## Part D & E — Full Model

In [ ]:
cfg = TransformerConfig(
    vocab_size=512, d_model=128, n_heads=4, n_layers=4,
    d_ff=512, max_seq_len=128, dropout=0.0,
    ffn_type='swiglu', norm_type='rmsnorm', pos_encoding='rope',
)
model = TransformerLM(cfg).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

tokens = torch.randint(0, cfg.vocab_size, (2, 32), device=device)
with torch.no_grad():
    logits = model(tokens)
print(f'Logits shape: {logits.shape}')

# Component breakdown
components = {}
for name, param in model.named_parameters():
    top = name.split('.')[0]
    components[top] = components.get(top, 0) + param.numel()

print()
for comp, count in sorted(components.items(), key=lambda x: -x[1]):
    print(f'  {comp:<20} {count:>8,}  ({100*count/total:.1f}%)')

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(list(components.values()), labels=list(components.keys()), autopct='%1.1f%%', startangle=90)
ax.set_title('Parameter distribution'); plt.tight_layout(); plt.show()

## Summary

We now have all the building blocks. The complete transformer block shows how:
- **Residual connections** → solve vanishing gradients
- **Pre-LN** → stable training without warmup tricks
- **RMSNorm** → efficient normalisation

**Next:** `../part3_innovations/07_efficient_attention_starter.ipynb`